# Script 15 v2 — AI Candidate Locked Holdout Evaluation

**Purpose:** evaluate AI forecasting candidates against Random Walk (RW) on the 2024–2025 holdout, instead of reporting RW as the only evaluated model.

## What this version does
- Rebuilds the point-in-time feature matrix from `Main set.xlsx`, following Script 02's weekday-only calendar and macro-availability proxies.
- Selects the ML configuration using **2023 development data only**.
- Freezes that configuration for the 2024–2025 holdout. Annual refitting uses only labels whose target dates have resolved before each year's start.
- Reports the RW benchmark, raw ML, shrunk ML, and a pre-specified online RW/ML hybrid.
- If TensorFlow is installed, also trains an LSTM/GRU challenger selected using 2023 development data only, then evaluates its frozen configuration on the holdout.
- Keeps EWMA94 as the volatility benchmark/fallback while the previous GARCH-family extreme-value warnings remain unresolved.

## Important methodological notes
- H1/H3/H7 mean 1, 3, and 7 **weekday observations ahead**, consistent with Script 02—not calendar days.
- Holdout scoring uses forecast origins and target dates within 2024–2025. This avoids using a model trained after a 2023 forecast origin to score a 2024 target.
- 2024 outcomes may enter the 2025 model's training set only after their target dates have resolved, as a pre-specified sequential/annual-refit design. Model/feature/hyperparameter selection remains frozen from 2023.
- This notebook evaluates AI challengers honestly; it does not guarantee that any model will beat RW. Do not tune the model based on these holdout results.
- The LSTM/GRU section is optional at runtime if TensorFlow is unavailable. If skipped, the output explicitly says so; do not claim DL holdout performance in that case.

**Save in `Research/notebooks/` and run all cells from top to bottom.**

## 0. Setup and paths

In [ ]:

from pathlib import Path
import warnings, json, time, math
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]
HOLDOUT_START = pd.Timestamp("2024-01-01")
HOLDOUT_END = pd.Timestamp("2025-12-31")
SEED = 42
SHRINKAGE_GRID = [0.0, 0.25, 0.50, 0.75, 1.0]
MODEL_KEYS = ["RIDGE_1", "EXTRA_TREES", "HIST_GB"]
TT_LAGS = [1, 2, 3, 5, 7, 14, 30]
ROLL_WINDOWS = [3, 5, 7, 14, 30]
VOL_WINDOWS = [5, 7, 14, 30]

GLOBAL_MARKET_VARS = ["DXY", "Crude_Oil_Price", "VIXCLS"]
POLICY_VARS = ["SDF_Rate", "SLF_Rate"]
PERIODIC_MACRO_VARS = [
    "Remittances_USD_Mn", "Tourism_USD_Mn", "FX_Reserves_USD_Mn",
    "Total_Exports", "Total_Imports", "CSE_Net_Inflow", "TBills_Inflow_Gross"
]
ANNUAL_MACRO_VARS = [
    "Direct_Tax", "Indirect_Tax", "Tax_Revenue_Total", "Non_Tax_Revenue",
    "Total_Revenue", "Political_Stability", "BOP", "GDP_USD", "Inflation"
]

def find_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "outputs").exists() and ((p / "Main set.xlsx").exists() or (p / "data" / "raw" / "Main set.xlsx").exists()):
            return p.resolve()
    return Path.cwd().resolve()

ROOT = find_root()
TABLE_DIR = ROOT / "outputs" / "tables"
PRED_DIR = ROOT / "outputs" / "predictions"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

def find_source():
    checks = [
        ROOT / "Main set.xlsx",
        ROOT / "data" / "raw" / "Main set.xlsx",
        ROOT / "data" / "processed" / "Main set.xlsx",
        Path.cwd() / "Main set.xlsx",
        Path("/mnt/data/Main set.xlsx"),
    ]
    for p in checks:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        "Main set.xlsx is required because it contains the macroeconomic predictors. "
        "Place it in the project root or data/raw/."
    )

SOURCE_PATH = find_source()
print("Project root:", ROOT)
print("Full feature source:", SOURCE_PATH)
print("Locked holdout:", HOLDOUT_START.date(), "to", HOLDOUT_END.date())


## 1. Rebuild point-in-time weekday features and H1/H3/H7 targets

In [ ]:

# Rebuild the documented point-in-time feature matrix from Main set.xlsx.
# IMPORTANT: weekday-only calendar, matching Script 02; no weekend carry-forward rows.

raw = pd.read_excel(SOURCE_PATH)
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce").dt.normalize()
if raw["Date"].isna().any():
    raise ValueError("Unparseable dates in Main set.xlsx.")
raw = raw.sort_values("Date").reset_index(drop=True)
if raw["Date"].duplicated().any():
    raise ValueError("Duplicate source dates detected in Main set.xlsx; resolve before modelling.")
required = ["Date"] + [f"{c}_{side}" for c in CURRENCIES for side in ["buy", "sell", "mid"]]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Apply the research's primary weekday calendar before constructing horizons.
model_df = raw.loc[raw["Date"].dt.dayofweek < 5].copy().sort_values("Date").reset_index(drop=True)

for c in CURRENCIES:
    for col in [f"{c}_buy", f"{c}_sell", f"{c}_mid"]:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce")
    model_df[f"{c}_quote_anomaly"] = (
        (model_df[f"{c}_buy"] > model_df[f"{c}_sell"]) |
        (model_df[f"{c}_buy"] <= 0) |
        (model_df[f"{c}_sell"] <= 0) |
        (model_df[f"{c}_mid"] <= 0)
    ).astype(int)
    model_df[f"{c}_buy_clean"] = model_df[f"{c}_buy"].astype(float)
    model_df[f"{c}_sell_clean"] = model_df[f"{c}_sell"].astype(float)
    bad = model_df[f"{c}_quote_anomaly"].eq(1)
    model_df.loc[bad, [f"{c}_buy_clean", f"{c}_sell_clean"]] = np.nan

def previous_change_block_value(series):
    s = series.copy()
    change = s.ne(s.shift(1)) | (s.isna() != s.shift(1).isna())
    block_id = change.cumsum()
    block_first = s.groupby(block_id).first()
    previous_block = block_first.shift(1)
    return block_id.map(previous_block)

# TT-derived, trailing features
for c in CURRENCIES:
    mid = model_df[f"{c}_mid"]
    buy = model_df[f"{c}_buy_clean"]
    sell = model_df[f"{c}_sell_clean"]
    model_df[f"{c}_spread"] = sell - buy
    model_df[f"{c}_spread_pct"] = 100 * (sell - buy) / mid
    model_df[f"{c}_pct_return"] = mid.pct_change()
    model_df[f"{c}_log_return"] = np.log(mid).diff()
    model_df[f"{c}_abs_log_return"] = model_df[f"{c}_log_return"].abs()
    for lag in TT_LAGS:
        model_df[f"{c}_mid_lag{lag}"] = mid.shift(lag)
    for w in ROLL_WINDOWS:
        model_df[f"{c}_mid_ma{w}"] = mid.rolling(w, min_periods=w).mean()
        model_df[f"{c}_mid_ema{w}"] = mid.ewm(span=w, adjust=False, min_periods=w).mean()
        model_df[f"{c}_mid_min{w}"] = mid.rolling(w, min_periods=w).min()
        model_df[f"{c}_mid_max{w}"] = mid.rolling(w, min_periods=w).max()
    for w in VOL_WINDOWS:
        model_df[f"{c}_logret_std{w}"] = model_df[f"{c}_log_return"].rolling(w, min_periods=w).std()

# Cross-currency features use only same-date/current or lagged TT quotes.
for c in CURRENCIES:
    for other in CURRENCIES:
        if other == c:
            continue
        model_df[f"{other}_mid_current_for_{c}"] = model_df[f"{other}_mid"]
        for lag in [1, 3, 7]:
            model_df[f"{other}_mid_lag{lag}_for_{c}"] = model_df[f"{other}_mid"].shift(lag)

# Conservative PIT transformations; raw contemporaneous macro columns are not model features.
for v in GLOBAL_MARKET_VARS + POLICY_VARS:
    if v in model_df.columns:
        model_df[f"{v}_PIT"] = pd.to_numeric(model_df[v], errors="coerce").shift(1)
for v in PERIODIC_MACRO_VARS:
    if v in model_df.columns:
        model_df[f"{v}_PIT"] = previous_change_block_value(pd.to_numeric(model_df[v], errors="coerce"))
for v in ANNUAL_MACRO_VARS:
    if v in model_df.columns:
        vals = pd.to_numeric(model_df[v], errors="coerce")
        by_year = model_df.assign(_year=model_df["Date"].dt.year, _val=vals).groupby("_year")["_val"].first().to_dict()
        model_df[f"{v}_PIT"] = model_df["Date"].dt.year.map(lambda y: by_year.get(y - 1, np.nan))

model_df["day_of_week"] = model_df["Date"].dt.dayofweek
model_df["month"] = model_df["Date"].dt.month
model_df["quarter"] = model_df["Date"].dt.quarter
model_df["day_of_month"] = model_df["Date"].dt.day
model_df["is_month_end"] = model_df["Date"].dt.is_month_end.astype(int)
model_df["is_month_start"] = model_df["Date"].dt.is_month_start.astype(int)
CALENDAR_FEATURES = ["day_of_week", "month", "quarter", "day_of_month", "is_month_end", "is_month_start"]

for h in HORIZONS:
    model_df[f"Target_Date_H{h}"] = model_df["Date"].shift(-h)
    for c in CURRENCIES:
        current = model_df[f"{c}_mid"]
        future = current.shift(-h)
        model_df[f"{c}_target_H{h}"] = future
        model_df[f"{c}_change_H{h}"] = future - current
        model_df[f"{c}_log_change_H{h}"] = np.log(future / current)

PIT_GLOBAL_POLICY = [f"{v}_PIT" for v in GLOBAL_MARKET_VARS + POLICY_VARS if f"{v}_PIT" in model_df.columns]
PIT_PERIODIC = [f"{v}_PIT" for v in PERIODIC_MACRO_VARS if f"{v}_PIT" in model_df.columns]
PIT_ANNUAL = [f"{v}_PIT" for v in ANNUAL_MACRO_VARS if f"{v}_PIT" in model_df.columns]
PIT_MACRO_ALL = PIT_GLOBAL_POLICY + PIT_PERIODIC + PIT_ANNUAL
FEATURE_SETS = {}
for c in CURRENCIES:
    own = [
        f"{c}_mid", f"{c}_buy_clean", f"{c}_sell_clean", f"{c}_spread",
        f"{c}_spread_pct", f"{c}_pct_return", f"{c}_log_return",
        f"{c}_abs_log_return", f"{c}_quote_anomaly"
    ]
    own += [f"{c}_mid_lag{lag}" for lag in TT_LAGS]
    for w in ROLL_WINDOWS:
        own += [f"{c}_mid_ma{w}", f"{c}_mid_ema{w}", f"{c}_mid_min{w}", f"{c}_mid_max{w}"]
    own += [f"{c}_logret_std{w}" for w in VOL_WINDOWS] + CALENDAR_FEATURES
    cross = []
    for other in CURRENCIES:
        if other != c:
            cross += [f"{other}_mid_current_for_{c}", f"{other}_mid_lag1_for_{c}",
                      f"{other}_mid_lag3_for_{c}", f"{other}_mid_lag7_for_{c}"]
    FEATURE_SETS[c] = {
        "TT_OWN": own,
        "TT_CROSS": own + cross,
        "FULL_PIT": own + cross + PIT_MACRO_ALL,
    }

if model_df["Date"].max() < HOLDOUT_END:
    raise ValueError(f"Source ends at {model_df['Date'].max().date()}, before holdout end.")
if model_df["Date"].max() > HOLDOUT_END:
    print("Note: source includes dates after the holdout; these will not be scored.")
print("Weekday rows:", len(model_df), "|", model_df["Date"].min().date(), "to", model_df["Date"].max().date())
print("2023 development rows:", int(model_df["Date"].dt.year.eq(2023).sum()))
print("2024–2025 holdout rows:", int(model_df["Date"].between(HOLDOUT_START, HOLDOUT_END).sum()))
print("Feature sets built for all 4 currencies.")


## 2. Model helpers

In [ ]:

# Candidate model definitions and evaluation helpers.
def build_model(key):
    if key == "RIDGE_1":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ])
    if key == "EXTRA_TREES":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=80, max_depth=7, min_samples_leaf=8,
                max_features=0.8, random_state=SEED, n_jobs=-1
            ))
        ])
    if key == "HIST_GB":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                learning_rate=0.04, max_iter=100, max_leaf_nodes=15,
                l2_regularization=2.0, random_state=SEED
            ))
        ])
    raise ValueError(key)

def rmse(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = np.isfinite(y) & np.isfinite(p)
    return float(np.sqrt(np.mean((y[m] - p[m]) ** 2))) if m.any() else np.nan

def mae(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = np.isfinite(y) & np.isfinite(p)
    return float(np.mean(np.abs(y[m] - p[m]))) if m.any() else np.nan

def dm_test(y, pred_model, pred_rw, horizon=1):
    y, pred_model, pred_rw = map(lambda x: np.asarray(x, float), (y, pred_model, pred_rw))
    m = np.isfinite(y) & np.isfinite(pred_model) & np.isfinite(pred_rw)
    y, pred_model, pred_rw = y[m], pred_model[m], pred_rw[m]
    n = len(y)
    if n < 20:
        return np.nan, np.nan
    d = (y - pred_rw) ** 2 - (y - pred_model) ** 2
    lag = max(0, int(horizon) - 1)
    lrv = np.var(d, ddof=1)
    for k in range(1, lag + 1):
        if len(d) <= k:
            break
        lrv += 2 * (1 - k / (lag + 1)) * np.cov(d[k:], d[:-k], ddof=1)[0, 1]
    if not np.isfinite(lrv) or lrv <= 0:
        return np.nan, np.nan
    stat = np.mean(d) / np.sqrt(lrv / n)
    return float(stat), float(2 * student_t.sf(abs(stat), df=n - 1))

def target_col(cur, h, form):
    return f"{cur}_{'change' if form == 'CHANGE' else 'log_change'}_H{h}"

def reconstruct(current, pred, form):
    current, pred = np.asarray(current, float), np.asarray(pred, float)
    return current + pred if form == "CHANGE" else current * np.exp(np.clip(pred, -20, 20))

def make_train_val(cur, h, form, family, window, year):
    features = [f for f in FEATURE_SETS[cur][family] if f in model_df.columns]
    target = target_col(cur, h, form)
    tdate = f"Target_Date_H{h}"
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year+1}-01-01")
    work = model_df.copy()
    work[target] = pd.to_numeric(work[target], errors="coerce")
    work[tdate] = pd.to_datetime(work[tdate], errors="coerce")
    train = work[(work["Date"] < start) & work[target].notna() & work[tdate].notna() & (work[tdate] < start)].copy()
    if window == "RECENT_5Y":
        train = train[train["Date"] >= start - pd.DateOffset(years=5)].copy()
    val_mask = (
        (work["Date"] >= start) & (work["Date"] < end) &
        work[target].notna() & work[tdate].notna() &
        (work[tdate] <= HOLDOUT_END)
    )
    # The 2023 configuration-selection fold may only use development targets.
    if year <= 2023:
        val_mask &= work[tdate] < end
    val = work.loc[val_mask].copy()
    # Drop features with almost no observed training history; imputation/scaling is then
    # fitted inside the training block only.
    features = [f for f in features if pd.to_numeric(train[f], errors="coerce").notna().sum() >= 20]
    return train, val, features, target



## 3. Select the ML model configuration using 2023 development data only

In [ ]:

# Select a frozen ML configuration per currency/horizon using development year 2023 only.
# Holdout outcomes are NOT used to select model, feature family, target form, training window, or shrinkage.

selection_rows = []
candidate_rows = []
SELECT_YEAR = 2023
FORMS = ["CHANGE", "LOG_CHANGE"]
FAMILIES = ["TT_OWN", "TT_CROSS", "FULL_PIT"]
WINDOWS = ["EXPANDING", "RECENT_5Y"]

for cur in CURRENCIES:
    for h in HORIZONS:
        print(f"Selecting from development only: {cur} H{h}", flush=True)
        train_base, val_base, _, _ = make_train_val(cur, h, "CHANGE", "TT_OWN", "EXPANDING", SELECT_YEAR)
        best = None
        best_val_score = np.inf
        best_val_preds = None

        for form in FORMS:
            for family in FAMILIES:
                for window in WINDOWS:
                    train, val, feats, target = make_train_val(cur, h, form, family, window, SELECT_YEAR)
                    if len(train) < 250 or len(val) < 60 or not feats:
                        continue
                    actual = val[f"{cur}_target_H{h}"].to_numpy(float)
                    origin_mid = val[f"{cur}_mid"].to_numpy(float)
                    rw_pred = origin_mid.copy()

                    for model_key in MODEL_KEYS:
                        try:
                            model = build_model(model_key)
                            model.fit(train[feats], train[target].astype(float))
                            raw = reconstruct(origin_mid, model.predict(val[feats]), form)
                            for lam in SHRINKAGE_GRID:
                                pred = rw_pred + lam * (raw - rw_pred)
                                score = rmse(actual, pred)
                                candidate_rows.append({
                                    "Currency": cur, "Horizon": h, "Selection_Year": SELECT_YEAR,
                                    "Target_Form": form, "Feature_Family": family, "Train_Window": window,
                                    "Model_Key": model_key, "Shrinkage_Lambda": lam,
                                    "N_Validation": int((np.isfinite(actual) & np.isfinite(pred)).sum()),
                                    "Validation_RMSE": score
                                })
                                if np.isfinite(score) and score < best_val_score:
                                    best_val_score = score
                                    best = {
                                        "Currency": cur, "Horizon": h, "Selection_Year": SELECT_YEAR,
                                        "Target_Form": form, "Feature_Family": family, "Train_Window": window,
                                        "Model_Key": model_key, "Shrinkage_Lambda": float(lam),
                                        "Development_2023_RMSE": float(score),
                                    }
                                    best_val_preds = pred.copy()
                        except Exception as e:
                            candidate_rows.append({
                                "Currency": cur, "Horizon": h, "Selection_Year": SELECT_YEAR,
                                "Target_Form": form, "Feature_Family": family, "Train_Window": window,
                                "Model_Key": model_key, "Shrinkage_Lambda": np.nan,
                                "N_Validation": len(val), "Validation_RMSE": np.nan,
                                "Error": repr(e)[:300]
                            })
        if best is None:
            # Safe, explicit fallback if the model selection cannot be fitted.
            best = {
                "Currency": cur, "Horizon": h, "Selection_Year": SELECT_YEAR,
                "Target_Form": "CHANGE", "Feature_Family": "TT_OWN", "Train_Window": "EXPANDING",
                "Model_Key": "RIDGE_1", "Shrinkage_Lambda": 0.0,
                "Development_2023_RMSE": np.nan, "Selection_Status": "FALLBACK_RW_NO_VALID_CANDIDATE"
            }
        else:
            best["Selection_Status"] = "SELECTED_ON_2023_DEVELOPMENT_ONLY"
        selection_rows.append(best)

selection_table = pd.DataFrame(selection_rows)
candidate_table = pd.DataFrame(candidate_rows)
display(selection_table)
print("Selection completed. All configurations were chosen using 2023 development data only.")


## 4. Generate holdout ML predictions with the frozen configuration

In [ ]:

# Generate 2024–2025 rolling-origin holdout forecasts using the frozen 2023 configuration.
# Model hyperparameters are frozen; annual refitting may use only training labels whose target dates
# have resolved before the relevant calendar year starts. This is a predeclared expanding/rolling update.

def fit_predict_year(cur, h, selection, year):
    form = selection["Target_Form"]
    family = selection["Feature_Family"]
    window = selection["Train_Window"]
    model_key = selection["Model_Key"]
    lam = float(selection["Shrinkage_Lambda"])
    train, val, feats, target = make_train_val(cur, h, form, family, window, year)
    if len(val) == 0:
        return pd.DataFrame()
    if len(train) < 250 or not feats:
        # A documented RW fallback; never fabricate an ML prediction.
        out = val[["Date", f"Target_Date_H{h}", f"{cur}_mid", f"{cur}_target_H{h}"]].copy()
        out = out.rename(columns={"Date": "Forecast_Origin", f"Target_Date_H{h}": "Target_Date",
                                  f"{cur}_mid": "Origin_Mid", f"{cur}_target_H{h}": "Actual"})
        out["Currency"] = cur
        out["Horizon"] = h
        out["RW_Prediction"] = out["Origin_Mid"]
        out["ML_Raw_Prediction"] = out["Origin_Mid"]
        out["ML_Prediction"] = out["Origin_Mid"]
        out["Frozen_Shrinkage_Lambda"] = 0.0
        out["Model_Status"] = "RW_FALLBACK_INSUFFICIENT_TRAINING"
        return out
    model = build_model(model_key)
    model.fit(train[feats], train[target].astype(float))
    origin_mid = val[f"{cur}_mid"].to_numpy(float)
    raw = reconstruct(origin_mid, model.predict(val[feats]), form)
    rw_pred = origin_mid.copy()
    pred = rw_pred + lam * (raw - rw_pred)
    out = pd.DataFrame({
        "Currency": cur,
        "Horizon": h,
        "Forecast_Origin": val["Date"].to_numpy(),
        "Target_Date": val[f"Target_Date_H{h}"].to_numpy(),
        "Actual": val[f"{cur}_target_H{h}"].to_numpy(float),
        "Origin_Mid": origin_mid,
        "RW_Prediction": rw_pred,
        "ML_Raw_Prediction": raw,
        "ML_Prediction": pred,
        "Frozen_Model": model_key,
        "Target_Form": form,
        "Feature_Family": family,
        "Train_Window": window,
        "Frozen_Shrinkage_Lambda": lam,
        "Training_Rows": len(train),
        "Model_Status": "OK"
    })
    # Score only origins and targets inside the locked period.
    out = out[
        (pd.to_datetime(out["Forecast_Origin"]) >= HOLDOUT_START) &
        (pd.to_datetime(out["Forecast_Origin"]) <= HOLDOUT_END) &
        (pd.to_datetime(out["Target_Date"]) >= HOLDOUT_START) &
        (pd.to_datetime(out["Target_Date"]) <= HOLDOUT_END)
    ].copy()
    return out

holdout_frames = []
for _, sel in selection_table.iterrows():
    cur, h = sel["Currency"], int(sel["Horizon"])
    for year in [2024, 2025]:
        print(f"Forecasting {cur} H{h}, {year}", flush=True)
        holdout_frames.append(fit_predict_year(cur, h, sel.to_dict(), year))

holdout_ml = pd.concat(holdout_frames, ignore_index=True)
if holdout_ml.empty:
    raise RuntimeError("No ML holdout forecasts generated. Check the source workbook and date coverage.")
if holdout_ml.duplicated(["Currency", "Horizon", "Forecast_Origin"]).any():
    raise RuntimeError("Duplicate holdout forecast origins detected.")
print("ML holdout forecast rows:", len(holdout_ml))
display(holdout_ml.head())


## 5. Optional LSTM/GRU challenger (Script 11-style)

In [ ]:

# Deep-learning challenger based on Script 11's LSTM/GRU correction design.
# Hyperparameters and shrinkage are selected using 2023 development data only.
# The same selected configuration is then frozen for 2024–2025.
RUN_DL = True
DL_SEQ_GRID = [7, 14, 30]
DL_FAMILIES = ["TT_OWN", "TT_CROSS"]
DL_KINDS = ["LSTM", "GRU"]
DL_SHRINK_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
DL_EPOCHS = 35
DL_BATCH = 32
DL_PATIENCE = 5
dl_predictions = pd.DataFrame()
dl_selection_rows = []
dl_candidate_rows = []
DL_AVAILABLE = False

try:
    import os, random
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    import tensorflow as tf
    from sklearn.preprocessing import StandardScaler
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.optimizers import Adam
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    DL_AVAILABLE = True
except Exception as e:
    print("TensorFlow unavailable; DL branch will be skipped. ML holdout evaluation can still run.")
    print("Reason:", repr(e))

def dl_net(kind, seq_len, n_features):
    tf.keras.backend.clear_session()
    recurrent = LSTM(24, dropout=0.1) if kind == "LSTM" else GRU(24, dropout=0.1)
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        recurrent,
        Dense(12, activation="relu"),
        Dropout(0.1),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss="mse")
    return model

def dl_prepare(cur, h, family, seq_len, year):
    features = [f for f in FEATURE_SETS[cur][family] if f in model_df.columns]
    target = f"{cur}_change_H{h}"
    tdate = f"Target_Date_H{h}"
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year+1}-01-01")
    work = model_df[["Date", target, tdate, f"{cur}_mid"] + features].copy().reset_index(drop=True)
    for f in features:
        work[f] = pd.to_numeric(work[f], errors="coerce")
    hist_mask = work["Date"] < start
    features = [f for f in features if work.loc[hist_mask, f].notna().sum() >= 20]
    if not features:
        raise ValueError(f"No usable DL features for {cur} H{h} {family}.")
    fit_rows = work.loc[hist_mask, features]
    imputer = SimpleImputer(strategy="median")
    Xfit = imputer.fit_transform(fit_rows)
    scaler = StandardScaler()
    scaler.fit(Xfit)
    Xall = scaler.transform(imputer.transform(work[features]))
    trX, trY, vaX, vaY, dates, mids, target_dates = [], [], [], [], [], [], []
    for i in range(seq_len - 1, len(work)):
        origin = work.loc[i, "Date"]
        td = work.loc[i, tdate]
        y = work.loc[i, target]
        if pd.isna(y) or pd.isna(td):
            continue
        seq = Xall[i-seq_len+1:i+1]
        if not np.isfinite(seq).all():
            continue
        if origin < start and td < start:
            trX.append(seq); trY.append(float(y))
        elif start <= origin < end and td <= HOLDOUT_END:
            # For the 2023 selection fold, do not include targets crossing into 2024.
            if year <= 2023 and td >= end:
                continue
            vaX.append(seq); vaY.append(float(y)); dates.append(origin)
            mids.append(float(work.loc[i, f"{cur}_mid"])); target_dates.append(td)
    return (np.asarray(trX, float), np.asarray(trY, float),
            np.asarray(vaX, float), np.asarray(vaY, float),
            pd.to_datetime(dates), np.asarray(mids, float), pd.to_datetime(target_dates), len(features))

def dl_fit_predict(cur, h, family, seq_len, kind, year):
    Xtr, ytr, Xv, yv, dates, mids, target_dates, nfeat = dl_prepare(cur, h, family, seq_len, year)
    if len(Xtr) < 250 or len(Xv) < 30:
        raise ValueError(f"Insufficient sequences for {cur} H{h} {year}: train={len(Xtr)}, val={len(Xv)}")
    y_scaler = StandardScaler()
    ytr_scaled = y_scaler.fit_transform(ytr.reshape(-1, 1)).ravel()
    model = dl_net(kind, seq_len, Xtr.shape[2])
    early = EarlyStopping(monitor="val_loss", patience=DL_PATIENCE, restore_best_weights=True)
    model.fit(Xtr, ytr_scaled, epochs=DL_EPOCHS, batch_size=DL_BATCH,
              validation_split=0.15, shuffle=False, callbacks=[early], verbose=0)
    pred_change = y_scaler.inverse_transform(model.predict(Xv, verbose=0).reshape(-1, 1)).ravel()
    return pd.DataFrame({
        "Currency": cur, "Horizon": h, "Forecast_Origin": dates, "Target_Date": target_dates,
        "Origin_Mid": mids, "Actual": mids + yv, "Raw_DL_Prediction": mids + pred_change,
        "DL_Feature_Family": family, "DL_Sequence_Length": seq_len, "DL_Model_Type": kind,
        "DL_Feature_Count": nfeat
    })

if RUN_DL and DL_AVAILABLE:
    for cur in CURRENCIES:
        for h in HORIZONS:
            print(f"Selecting DL config on 2023 only: {cur} H{h}", flush=True)
            best_cfg = None
            best_score = np.inf
            for family in DL_FAMILIES:
                for seq_len in DL_SEQ_GRID:
                    for kind in DL_KINDS:
                        try:
                            val = dl_fit_predict(cur, h, family, seq_len, kind, 2023)
                            rwv = val["Origin_Mid"].to_numpy(float)
                            actual = val["Actual"].to_numpy(float)
                            raw = val["Raw_DL_Prediction"].to_numpy(float)
                            for lam in DL_SHRINK_GRID:
                                pred = rwv + lam * (raw - rwv)
                                score = rmse(actual, pred)
                                dl_candidate_rows.append({
                                    "Currency": cur, "Horizon": h, "Selection_Year": 2023,
                                    "Feature_Family": family, "Sequence_Length": seq_len,
                                    "Model_Type": kind, "Shrinkage_Lambda": lam,
                                    "N_Validation": len(val), "Validation_RMSE": score
                                })
                                if np.isfinite(score) and score < best_score:
                                    best_score = score
                                    best_cfg = {
                                        "Currency": cur, "Horizon": h, "Selection_Year": 2023,
                                        "Feature_Family": family, "Sequence_Length": seq_len,
                                        "Model_Type": kind, "Shrinkage_Lambda": lam,
                                        "Development_2023_RMSE": score, "Selection_Status": "SELECTED_ON_2023_DEVELOPMENT_ONLY"
                                    }
                        except Exception as e:
                            dl_candidate_rows.append({
                                "Currency": cur, "Horizon": h, "Selection_Year": 2023,
                                "Feature_Family": family, "Sequence_Length": seq_len,
                                "Model_Type": kind, "Shrinkage_Lambda": np.nan,
                                "N_Validation": 0, "Validation_RMSE": np.nan, "Error": repr(e)[:300]
                            })
            if best_cfg is None:
                dl_selection_rows.append({
                    "Currency": cur, "Horizon": h, "Selection_Year": 2023,
                    "Feature_Family": "TT_OWN", "Sequence_Length": 7, "Model_Type": "LSTM",
                    "Shrinkage_Lambda": 0.0, "Development_2023_RMSE": np.nan,
                    "Selection_Status": "NO_VALID_DL_CONFIGURATION"
                })
            else:
                dl_selection_rows.append(best_cfg)

    dl_selection = pd.DataFrame(dl_selection_rows)
    display(dl_selection)
    holdout_dl_frames = []
    for _, sel in dl_selection.iterrows():
        if sel["Selection_Status"] != "SELECTED_ON_2023_DEVELOPMENT_ONLY":
            continue
        cur, h = sel["Currency"], int(sel["Horizon"])
        for year in [2024, 2025]:
            print(f"Fitting frozen DL config: {cur} H{h}, {year}", flush=True)
            try:
                out = dl_fit_predict(cur, h, sel["Feature_Family"], int(sel["Sequence_Length"]), sel["Model_Type"], year)
                lam = float(sel["Shrinkage_Lambda"])
                out["DL_Prediction"] = out["Origin_Mid"] + lam * (out["Raw_DL_Prediction"] - out["Origin_Mid"])
                out["DL_Frozen_Shrinkage_Lambda"] = lam
                out["DL_Training_Year"] = year
                # Restrict to origin and target inside the locked holdout.
                out = out[
                    (out["Forecast_Origin"] >= HOLDOUT_START) &
                    (out["Forecast_Origin"] <= HOLDOUT_END) &
                    (out["Target_Date"] >= HOLDOUT_START) &
                    (out["Target_Date"] <= HOLDOUT_END)
                ].copy()
                holdout_dl_frames.append(out)
            except Exception as e:
                print(f"DL forecast skipped for {cur} H{h} {year}: {repr(e)}")
    if holdout_dl_frames:
        dl_predictions = pd.concat(holdout_dl_frames, ignore_index=True)
    print("DL holdout rows:", len(dl_predictions))
else:
    dl_selection = pd.DataFrame()
    print("DL branch skipped; the ML challenger remains available.")


## 6. Pre-specified online RW/ML hybrid

In [ ]:

# Pre-specified online RW/ML hybrid.
# At each origin, weights use only earlier forecasts whose target dates have resolved.
# If ML has not shown better recent RMSE, use RW; otherwise blend toward ML conservatively.

ONLINE_WINDOW = 60
MIN_RESOLVED = 20
MAX_ML_WEIGHT = 0.75
hybrid_frames = []

for (cur, h), task in holdout_ml.groupby(["Currency", "Horizon"], sort=True):
    task = task.sort_values("Forecast_Origin").reset_index(drop=True).copy()
    preds = []
    weights = []
    for i, row in task.iterrows():
        origin = pd.Timestamp(row["Forecast_Origin"])
        hist = task.iloc[:i].copy()
        hist = hist[(pd.to_datetime(hist["Target_Date"]) <= origin) &
                    (pd.to_datetime(hist["Forecast_Origin"]) < origin)].tail(ONLINE_WINDOW)
        alpha = float(row["Frozen_Shrinkage_Lambda"]) if row["Model_Status"] == "OK" else 0.0
        if len(hist) >= MIN_RESOLVED:
            rw_rmse = rmse(hist["Actual"].to_numpy(float), hist["RW_Prediction"].to_numpy(float))
            ml_rmse = rmse(hist["Actual"].to_numpy(float), hist["ML_Prediction"].to_numpy(float))
            if np.isfinite(rw_rmse) and rw_rmse > 0 and np.isfinite(ml_rmse) and ml_rmse < rw_rmse:
                alpha = min(MAX_ML_WEIGHT, max(0.0, 1.0 - ml_rmse / rw_rmse))
            else:
                alpha = 0.0
        hybrid = float(row["RW_Prediction"] + alpha * (row["ML_Prediction"] - row["RW_Prediction"]))
        preds.append(hybrid)
        weights.append(alpha)
    task["AI_Online_Hybrid_Prediction"] = preds
    task["Online_ML_Weight"] = weights
    hybrid_frames.append(task)

holdout_predictions = pd.concat(hybrid_frames, ignore_index=True)
if not dl_predictions.empty:
    dl_keep = ["Currency", "Horizon", "Forecast_Origin", "Target_Date", "Raw_DL_Prediction", "DL_Prediction"]
    dl_keep = [c for c in dl_keep if c in dl_predictions.columns]
    holdout_predictions = holdout_predictions.merge(
        dl_predictions[dl_keep].drop_duplicates(["Currency", "Horizon", "Forecast_Origin", "Target_Date"]),
        on=["Currency", "Horizon", "Forecast_Origin", "Target_Date"],
        how="left",
        validate="one_to_one"
    )
print("Hybrid forecasts generated with a resolved-error-only rule.")


## 7. Optional regime-gated DL hybrid

In [ ]:

# Optional regime gate for the LSTM/GRU challenger, following Script 11B's principle.
# The gate parameters come from the 2023 development selection file when available.
# If that file is absent, documented conservative defaults are used; holdout scores never tune them.
if "DL_Prediction" in holdout_predictions.columns and holdout_predictions["DL_Prediction"].notna().any():
    gate_params_path = TABLE_DIR / "11B_regime_gated_DL_strategy_selections.csv"
    gate_params = {}
    if gate_params_path.is_file():
        try:
            dev_params = pd.read_csv(gate_params_path)
            dev_params = dev_params[dev_params["Outer_Year"].eq(2023)]
            for _, r in dev_params.iterrows():
                gate_params[(r["Currency"], int(r["Horizon"]))] = {
                    "vol_q": float(r["vol_q"]),
                    "err_window": int(r["err_window"]),
                    "dl_weight": float(r["dl_weight"]),
                    "margin": float(r["margin"]),
                }
        except Exception as e:
            print("Could not read 11B development gate parameters; documented defaults will be used:", repr(e))
    default_gate = {"vol_q": 0.70, "err_window": 40, "dl_weight": 0.50, "margin": 0.005}
    vol_map_by_currency = {}
    for cur in CURRENCIES:
        vol_series = model_df.set_index("Date")[f"{cur}_log_return"].rolling(20, min_periods=10).std()
        vol_map_by_currency[cur] = vol_series

    gated_frames = []
    for (cur, h), task in holdout_predictions.groupby(["Currency", "Horizon"], sort=True):
        task = task.sort_values("Forecast_Origin").reset_index(drop=True).copy()
        params = gate_params.get((cur, int(h)), default_gate)
        vol_series = vol_map_by_currency[cur]
        current_vol = pd.to_datetime(task["Forecast_Origin"]).map(vol_series)
        task["DL_Gate_VOL20"] = current_vol.to_numpy(float)
        active_flags, gated_preds = [], []
        for i, row in task.iterrows():
            origin = pd.Timestamp(row["Forecast_Origin"])
            past_vol = vol_series.loc[vol_series.index < origin].dropna()
            threshold = float(past_vol.quantile(params["vol_q"])) if len(past_vol) >= 20 else np.nan
            hist = task.iloc[:i].copy()
            hist = hist[
                (pd.to_datetime(hist["Forecast_Origin"]) < origin) &
                (pd.to_datetime(hist["Target_Date"]) <= origin) &
                np.isfinite(pd.to_numeric(hist["Actual"], errors="coerce")) &
                np.isfinite(pd.to_numeric(hist["AI_Online_Hybrid_Prediction"], errors="coerce")) &
                np.isfinite(pd.to_numeric(hist["DL_Prediction"], errors="coerce"))
            ].tail(params["err_window"])
            use_dl = False
            if len(hist) >= 20 and np.isfinite(row["DL_Gate_VOL20"]) and np.isfinite(threshold):
                base_rmse = rmse(hist["Actual"].to_numpy(float), hist["AI_Online_Hybrid_Prediction"].to_numpy(float))
                dl_rmse = rmse(hist["Actual"].to_numpy(float), hist["DL_Prediction"].to_numpy(float))
                advantage = (base_rmse - dl_rmse) / base_rmse if np.isfinite(base_rmse) and base_rmse > 0 and np.isfinite(dl_rmse) else -np.inf
                use_dl = bool(row["DL_Gate_VOL20"] >= threshold and advantage > params["margin"])
            base_pred = float(row["AI_Online_Hybrid_Prediction"])
            dl_pred = float(row["DL_Prediction"]) if np.isfinite(row["DL_Prediction"]) else np.nan
            if use_dl and np.isfinite(dl_pred):
                final_pred = (1.0 - params["dl_weight"]) * base_pred + params["dl_weight"] * dl_pred
            else:
                final_pred = base_pred
            active_flags.append(use_dl)
            gated_preds.append(final_pred)
        task["AI_Regime_Gated_DL_Hybrid_Prediction"] = gated_preds
        task["DL_Gate_Activated"] = active_flags
        task["DL_Gate_Quantile"] = params["vol_q"]
        task["DL_Gate_Error_Window"] = params["err_window"]
        task["DL_Gate_Blend_Weight"] = params["dl_weight"]
        task["DL_Gate_Advantage_Margin"] = params["margin"]
        gated_frames.append(task)
    holdout_predictions = pd.concat(gated_frames, ignore_index=True)
    print("Regime-gated DL hybrid created using resolved past errors only.")
else:
    print("Regime-gated DL hybrid skipped because no DL holdout predictions are available.")


## 8. Point-forecast evaluation and benchmark comparisons

In [ ]:

# Score the benchmark and AI challengers. The holdout is for evaluation, not model selection.
def mape_percent(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = np.isfinite(y) & np.isfinite(p) & (np.abs(y) > 1e-10)
    return float(100 * np.mean(np.abs((y[m] - p[m]) / y[m]))) if m.any() else np.nan

def smape_percent(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    den = np.abs(y) + np.abs(p)
    m = np.isfinite(y) & np.isfinite(p) & (den > 1e-10)
    return float(200 * np.mean(np.abs(y[m] - p[m]) / den[m])) if m.any() else np.nan

model_cols = {
    "Random_Walk_Benchmark": "RW_Prediction",
    "AI_ML_Raw": "ML_Raw_Prediction",
    "AI_ML_Shrunk": "ML_Prediction",
    "AI_Online_RW_ML_Hybrid": "AI_Online_Hybrid_Prediction",
}
if "Raw_DL_Prediction" in holdout_predictions.columns:
    model_cols["AI_DL_Raw"] = "Raw_DL_Prediction"
if "DL_Prediction" in holdout_predictions.columns:
    model_cols["AI_DL_Shrunk"] = "DL_Prediction"
if "AI_Regime_Gated_DL_Hybrid_Prediction" in holdout_predictions.columns:
    model_cols["AI_Regime_Gated_DL_Hybrid"] = "AI_Regime_Gated_DL_Hybrid_Prediction"
score_rows = []
for (cur, h), g in holdout_predictions.groupby(["Currency", "Horizon"], sort=True):
    y = g["Actual"].to_numpy(float)
    rwv = g["RW_Prediction"].to_numpy(float)
    rw_rmse = rmse(y, rwv)
    for model_name, col in model_cols.items():
        p = g[col].to_numpy(float)
        dm_stat, dm_p = (np.nan, np.nan) if model_name == "Random_Walk_Benchmark" else dm_test(y, p, rwv, int(h))
        model_rmse = rmse(y, p)
        score_rows.append({
            "Currency": cur, "Horizon": int(h), "Model": model_name,
            "N": int((np.isfinite(y) & np.isfinite(p)).sum()),
            "Target_Start": pd.to_datetime(g["Target_Date"]).min().date().isoformat(),
            "Target_End": pd.to_datetime(g["Target_Date"]).max().date().isoformat(),
            "RMSE": model_rmse, "MAE": mae(y, p),
            "MAPE_Percent": mape_percent(y, p), "sMAPE_Percent": smape_percent(y, p),
            "Forecast_Skill_vs_RW": (1 - model_rmse / rw_rmse) if np.isfinite(model_rmse) and rw_rmse > 0 else np.nan,
            "Theils_U": (model_rmse / rw_rmse) if np.isfinite(model_rmse) and rw_rmse > 0 else np.nan,
            "DM_Statistic_vs_RW": dm_stat, "DM_pvalue_vs_RW": dm_p,
            "Mean_Error_Actual_minus_Prediction": float(np.nanmean(y - p)),
            "Mean_Online_ML_Weight": float(g["Online_ML_Weight"].mean()) if "Online_ML_Weight" in g else np.nan,
        })
point_scores = pd.DataFrame(score_rows)

# Holm adjustment within each challenger across the 12 currency/horizon tasks.
def holm_adjust(p_values):
    p = np.asarray(p_values, float)
    out = np.full(len(p), np.nan)
    valid_idx = np.where(np.isfinite(p))[0]
    if len(valid_idx) == 0:
        return out
    order = valid_idx[np.argsort(p[valid_idx])]
    m = len(order)
    adjusted = []
    running = 0.0
    for rank, idx in enumerate(order):
        val = min(1.0, (m - rank) * p[idx])
        running = max(running, val)
        adjusted.append((idx, running))
    for idx, val in adjusted:
        out[idx] = val
    return out

point_scores["DM_pvalue_Holm_within_model"] = np.nan
for model_name in [m for m in point_scores["Model"].unique() if m != "Random_Walk_Benchmark"]:
    mask = point_scores["Model"].eq(model_name)
    point_scores.loc[mask, "DM_pvalue_Holm_within_model"] = holm_adjust(point_scores.loc[mask, "DM_pvalue_vs_RW"].to_numpy())

display(point_scores.sort_values(["Currency", "Horizon", "Model"]))


## 9. EWMA94 volatility benchmark

In [ ]:

# Volatility fallback: EWMA94 benchmark, evaluated on the same weekday-origin convention.
# This does not claim that EWMA is an AI model; it is the volatility baseline until a candidate passes its audit.

EWMA_LAMBDA = 0.94
vol_rows = []
for cur in CURRENCIES:
    mid = model_df[f"{cur}_mid"].to_numpy(float)
    dates = pd.to_datetime(model_df["Date"]).to_numpy(dtype="datetime64[ns]")
    ret = (100.0 * np.log(pd.Series(mid) / pd.Series(mid).shift(1))).to_numpy(dtype=float)
    r2 = ret ** 2
    ewma = np.full(len(model_df), np.nan)
    for i in range(len(model_df)):
        if not np.isfinite(r2[i]):
            continue
        if i == 0 or not np.isfinite(ewma[i-1]):
            hist = r2[:i+1]
            hist = hist[np.isfinite(hist)]
            ewma[i] = float(np.mean(hist)) if len(hist) else np.nan
        else:
            ewma[i] = EWMA_LAMBDA * ewma[i-1] + (1-EWMA_LAMBDA) * r2[i]
    for h in HORIZONS:
        for i in range(len(model_df) - h):
            origin = pd.Timestamp(dates[i])
            target = pd.Timestamp(dates[i+h])
            if origin < HOLDOUT_START or origin > HOLDOUT_END or target < HOLDOUT_START or target > HOLDOUT_END:
                continue
            future = ret[i+1:i+h+1]
            rv = float(np.sum(future**2)) if len(future) == h and np.isfinite(future).all() else np.nan
            vf = float(h * ewma[i]) if np.isfinite(ewma[i]) and ewma[i] > 0 else np.nan
            vol_rows.append({
                "Currency": cur, "Horizon": h, "Forecast_Origin": origin, "Target_Date": target,
                "Realized_Variance": rv, "Variance_Forecast": vf, "Selected_Model": "EWMA94",
                "EWMA_Lambda": EWMA_LAMBDA
            })
vol_predictions = pd.DataFrame(vol_rows)

def qlike(rv, vf):
    rv, vf = np.asarray(rv, float), np.asarray(vf, float)
    m = np.isfinite(rv) & np.isfinite(vf) & (rv >= 0) & (vf > 0)
    out = np.full(len(rv), np.nan)
    out[m] = np.log(vf[m]) + np.maximum(rv[m], 1e-10) / vf[m]
    return out

vol_score_rows = []
for (cur, h), g in vol_predictions.groupby(["Currency", "Horizon"], sort=True):
    rv = g["Realized_Variance"].to_numpy(float)
    vf = g["Variance_Forecast"].to_numpy(float)
    valid = np.isfinite(rv) & np.isfinite(vf) & (rv >= 0) & (vf > 0)
    q = qlike(rv, vf)
    vol_score_rows.append({
        "Currency": cur, "Horizon": int(h), "Model": "EWMA94_Benchmark",
        "N": int(valid.sum()), "Invalid_Rows": int(len(g)-valid.sum()),
        "QLIKE": float(np.nanmean(q)) if np.isfinite(q).any() else np.nan,
        "Variance_MSE": float(np.mean((rv[valid]-vf[valid])**2)) if valid.any() else np.nan,
        "Min_Forecast_Variance": float(np.nanmin(vf)) if np.isfinite(vf).any() else np.nan,
        "Max_Forecast_Variance": float(np.nanmax(vf)) if np.isfinite(vf).any() else np.nan,
    })
vol_scores = pd.DataFrame(vol_score_rows)
display(vol_scores)


## 10. Quality gate and outputs

In [ ]:

# Final audit and outputs.
quality_rows = []
def gate(name, passed, evidence):
    quality_rows.append({"Check": name, "Status": "PASS" if bool(passed) else "FAIL", "Evidence": str(evidence)})

gate("Main set workbook found", SOURCE_PATH.is_file(), SOURCE_PATH)
gate("Weekday calendar used", bool((model_df["Date"].dt.dayofweek < 5).all()), f"{len(model_df)} weekday rows")
gate("Full 2024–2025 source coverage", model_df["Date"].max() >= HOLDOUT_END, model_df["Date"].max().date())
gate("All 12 currency/horizon tasks scored", point_scores[["Currency", "Horizon"]].drop_duplicates().shape[0] == 12, point_scores[["Currency", "Horizon"]].drop_duplicates().shape[0])
gate("AI candidate forecasts present", {"AI_ML_Raw", "AI_ML_Shrunk", "AI_Online_RW_ML_Hybrid"}.issubset(set(point_scores["Model"])), sorted(point_scores["Model"].unique()))
gate("Holdout origins and targets only", ((pd.to_datetime(holdout_predictions["Forecast_Origin"]) >= HOLDOUT_START) & (pd.to_datetime(holdout_predictions["Forecast_Origin"]) <= HOLDOUT_END) & (pd.to_datetime(holdout_predictions["Target_Date"]) >= HOLDOUT_START) & (pd.to_datetime(holdout_predictions["Target_Date"]) <= HOLDOUT_END)).all(), "Origins and target dates restricted to 2024–2025")
gate("No duplicate point forecast keys", not holdout_predictions.duplicated(["Currency", "Horizon", "Forecast_Origin", "Target_Date"]).any(), "Currency/Horizon/Origin/Target uniqueness")
gate("Model selection frozen before holdout", selection_table["Selection_Year"].eq(2023).all(), "All ML configuration selection used 2023 development only")
gate("No holdout model-selection stage", True, "2024–2025 used only for evaluation and prespecified annual refit")
gate("Volatility fallback scores available", len(vol_scores) == 12, f"{len(vol_scores)}/12 EWMA tasks")
quality_gate = pd.DataFrame(quality_rows)

# Outputs
outputs = {
    TABLE_DIR / "15_ai_holdout_model_selection_2023.csv": selection_table,
    TABLE_DIR / "15_ai_holdout_candidate_grid_2023.csv": candidate_table,
    TABLE_DIR / "15_ai_holdout_dl_selection_2023.csv": dl_selection,
    TABLE_DIR / "15_ai_holdout_dl_candidate_grid_2023.csv": pd.DataFrame(dl_candidate_rows),
    TABLE_DIR / "15_ai_holdout_point_task_scores.csv": point_scores,
    TABLE_DIR / "15_ai_holdout_volatility_task_scores.csv": vol_scores,
    TABLE_DIR / "15_ai_holdout_quality_gate.csv": quality_gate,
    PRED_DIR / "15_ai_holdout_predictions.csv": holdout_predictions,
    PRED_DIR / "15_ai_holdout_dl_predictions.csv": dl_predictions,
    PRED_DIR / "15_ai_holdout_volatility_predictions.csv": vol_predictions,
}
for path, frame in outputs.items():
    frame.to_csv(path, index=False)

print("="*78)
print("SCRIPT 15 — AI CANDIDATE HOLDOUT EVALUATION")
print("="*78)
print("Selection year: 2023 development only")
print("Deep-learning branch:", "AVAILABLE" if (not dl_predictions.empty) else "SKIPPED/UNAVAILABLE")
print("Holdout: 2024–2025; weekdays; model config frozen before holdout")
print("Point prediction rows:", len(holdout_predictions))
print("Quality checks:", int(quality_gate["Status"].eq("PASS").sum()), "/", len(quality_gate))
print("\nPoint scores:")
print(point_scores.to_string(index=False))
print("\nQuality gate:")
print(quality_gate.to_string(index=False))
if quality_gate["Status"].ne("PASS").any():
    raise RuntimeError("Quality gate failed. Review 15_ai_holdout_quality_gate.csv before interpreting scores.")
print("\nCompleted. Do not tune model settings based on the holdout results.")
print("IMPORTANT: This version evaluates raw and shrunk ML, an online RW/ML hybrid, and (when TensorFlow is available) an LSTM/GRU challenger. EWMA94 remains the volatility benchmark; it does not claim a GARCH/EGARCH challenger won.")
